In [1]:
# Base types for the contraction tree
from tnco_core import ContractionTree, Node, Bitset

# The cost model to use. The format is SimpleCostModel_CostType_WidthType.
# The type float32 should be enough for small tensors. In case more
# precision is needed, it is recommended to keep WidthType fixed and
# increase CostType.
from tnco_core.optimize.finite_width.cost_model import (
    SimpleCostModel_float32_float32,
)

# The optimizer to use. The format is Optimizer_CostType_WidthType.
# The type float32 should be enough for small tensors. In case more
# precision is needed, it is recommended to keep WidthType fixed and
# increase CostType.
from tnco_core.optimize.finite_width.greedy import Optimizer_float32_float32

# The acceptance probability to use for the optimization. The format is
# MetropolisHastings_CostType. The type float32 should be enough for
# small tensors. In case more precision is needed, it is recommended to
# increase CostType.
from tnco_core.optimize.prob import MetropolisHastings_float32

# Get contraction from tensor network contraction
from tnco_core.utils import get_contraction

In [2]:
# Nodes of the full contraction tree. Each node has the format
# 
#   Node((child_1, child_2), parent)
#
# where child_1, child_2, and parent are integers representing the
# position in the list. A leaf is represented as:
#
#   leaf = Node((-1, -1), parent)
#
# while a root is represented as:
#
#   root = Node((child_1, child_2), -1)
#
# Each integer must be a valid location in the list 'nodes'.
# For instance, the following example represents the contraction
# tree:
#                    Node((2, 3), -1)
#                     /            \
#        Node((0, 1), 4)         Node((-1, -1), 4)
#             /         \
#  Node((-1, -1), 3)   Node((-1, -1), 3)
#
# The contraction tree must be organized so that all leaves are
# at the beginning of the list, while the root is the last node.
nodes = [
    Node((-1, -1), 3),
    Node((-1, -1), 3),
    Node((-1, -1), 4),
    Node((0, 1), 4),
    Node((2, 3), -1),
]

# Indices are represented as bitstrings. An index is present in
# the tensor if its corresponding bitstring has a one in that
# location. 'Bitset' can be initialized with either a string of
# zeros and ones, or by location.
ts_inds = [
    Bitset("11100"),
    Bitset((1, 2, 3), 5),
    Bitset((0, 2, 4), 5),
    Bitset((0, 2, 3), 5),
    Bitset((2, 3, 4), 5),
]

# Dimensions of each index. If all dimensions are the same, 'dims'
# can be a single integer
dims = 2

# Build the contraction tree from the list of nodes, tensor indices,
# and dimensions
ctree = ContractionTree(nodes, ts_inds, dims)
print(f'{ctree=}')

# Get cost model (this can be initialized only once)
cost_model = SimpleCostModel_float32_float32(2)
print(f'{cost_model=}')

# Get optimizer
optimizer = Optimizer_float32_float32(ctree, cost_model)
print(f'{optimizer=}')

# Update the contraction tree to minimize the contraction cost
optimizer.update(MetropolisHastings_float32(0.12), update_slices=True)

# Get optimized contraction tree
min_ctree = optimizer.min_ctree
min_slices = optimizer.min_slices.positions()
print(f'{min_ctree=}')
print(f'{min_slices=}')

# Contraction path. The contraction is in the format:
# (child_0, child_1, node_pos)
# ...
# meaning that 'node_pos' is obtained by contracting
# 'child_0' with 'child_1'.
contraction_path = get_contraction(min_ctree)
print(f'{contraction_path=}')

ctree=<tnco_core.ContractionTree object at 0x7fa8a17c3370>
cost_model=SimpleCostModel(max_width=2, width_type=float32, cost_type=float32)
optimizer=<tnco_core.optimize.finite_width.greedy.Optimizer_float32_float32 object at 0x7fa8a17c32f0>
min_ctree=<tnco_core.ContractionTree object at 0x7fa8a27129b0>
min_slices=[2]
contraction_path=[[0, 1, 3], [2, 3, 4]]
